# GEAP Agent Registry — Discovery, Governance & Hub-and-Spoke Across Projects (SDK-First Deep-Dive)

A companion to `platform_sdk_demo.ipynb`. The **Agent Registry** is the fleet catalog: it registers MCP servers and agents, resolves them by name (replacing hardcoded URLs), and governs connections between them. This notebook goes deeper — including a **hub-and-spoke** setup where a **hub** project automatically **discovers and searches agents deployed in a separate spoke project** across the project boundary, following the reference [`jswortz/hub-spoke-agents-gcp-26`](https://github.com/jswortz/hub-spoke-agents-gcp-26).

**Legend.** ✅ runs live (in-process discovery / read-only queries). 🔒 shown-but-guarded: the mutating gcloud/REST call is printed and only executed when `GEAP_RUN_REGISTRY=1`. 🔧 marks custom/infra.

> Hub-and-spoke uses **App Hub** to establish the boundary (host + service projects), then Agent Registry `search` to query across it. Hub = your project; spoke = **`agent-spoke`**. Read-only `search` / `discovered-workloads list` are attempted live; boundary changes are guarded.

## Setup

In [1]:
import os, json, shlex, subprocess
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

from src.config import GCP_PROJECT_ID, GCP_REGION, AGENT_REGISTRY_LOCATION, SEARCH_MCP_SERVER

os.environ["CLOUDSDK_CORE_DISABLE_PROMPTS"] = "1"   # never hang on an alpha-component install prompt

HUB_PROJECT   = GCP_PROJECT_ID                       # the registry hub
SPOKE_PROJECT = os.environ.get("AGENT_SPOKE_PROJECT_ID", "agent-spoke")
REGION        = AGENT_REGISTRY_LOCATION              # registry search is single-region (no wildcards)
RUN_REGISTRY  = os.environ.get("GEAP_RUN_REGISTRY") == "1"   # gate mutations (create/attach/register)

def run_or_show(cmd, live: bool, timeout: int = 90):
    """🔧 Print a command; execute it (never raising) only when `live` is True."""
    printable = cmd if isinstance(cmd, str) else " ".join(shlex.quote(c) for c in cmd)
    print(f"$ {printable}")
    if not live:
        print("  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real")
        return None
    try:
        out = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout)
        print((out.stdout or out.stderr)[-2000:] or "  (no output)")
        return out
    except Exception as e:
        print(f"  (skipped/failed: {type(e).__name__}: {e})")
        return None

print("hub:", HUB_PROJECT, "| spoke:", SPOKE_PROJECT, "| region:", REGION)
print("guard -> RUN_REGISTRY:", RUN_REGISTRY)

hub: wortz-project-352116 | spoke: agent-spoke | region: us-central1
guard -> RUN_REGISTRY: False


## Phase 1 — The registry model + discovery
📖 [`src/registry.py`](https://github.com/jswortz/geap-tour/blob/main/src/registry.py) · [workshop guide Session 3](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md)

You register **Services**; the registry then exposes read-only **Agent**, **McpServer**, and **Endpoint** projections. Agents discover tools by registered name via `AgentRegistry.get_mcp_toolset(...)` — no hardcoded URLs. The repo helper `get_mcp_tools` falls back to a direct Cloud Run URL if the registry entry isn't found, so this runs offline.

In [2]:
from src.registry import get_mcp_tools

print("MCP server resource name (from src/config.py):")
print(" ", SEARCH_MCP_SERVER)

toolset = get_mcp_tools(SEARCH_MCP_SERVER)   # registry discovery, with direct-URL fallback
print("\nresolved toolset:", type(toolset).__name__)
print("(registry lookup routes through the gateway for governance; the fallback URL bypasses it)")

/home/admin_jwortz_altostrat_com/geap-tour/.venv/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


MCP server resource name (from src/config.py):
  projects/wortz-project-352116/locations/us-central1/agentRegistries/default/mcpServers/search-mcp



resolved toolset: McpToolset
(registry lookup routes through the gateway for governance; the fallback URL bypasses it)


## Phase 2 — Register an MCP server — 🔒 guarded
📖 [workshop guide §3.1](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md) · [toolspecs](https://github.com/jswortz/geap-tour/blob/main/scripts/toolspecs)

A **Service** is created from a **toolspec** (the JSON that advertises the server's tools + input schemas + `readOnlyHint`/`isDestructive` annotations) and an interface URL. Those annotations are exactly what the gateway's CEL policies key off (see `gateway_sdk_demo.ipynb`).

In [3]:
# Inspect the toolspec that will be registered:
with open("scripts/toolspecs/search_toolspec.json") as f:
    spec = json.load(f)
print("tools advertised:", [t["name"] for t in spec["tools"]])

run_or_show([
    "gcloud", "alpha", "agent-registry", "services", "create", "search-mcp",
    "--project", HUB_PROJECT, "--location", REGION, "--display-name", "search-mcp",
    "--mcp-server-spec-type=tool-spec",
    "--mcp-server-spec-content=scripts/toolspecs/search_toolspec.json",
    "--interfaces=url=https://search-mcp-xxxx-uc.a.run.app/mcp,protocolBinding=JSONRPC",
], RUN_REGISTRY)

tools advertised: ['search_flights', 'search_hotels']
$ gcloud alpha agent-registry services create search-mcp --project wortz-project-352116 --location us-central1 --display-name search-mcp --mcp-server-spec-type=tool-spec --mcp-server-spec-content=scripts/toolspecs/search_toolspec.json --interfaces=url=https://search-mcp-xxxx-uc.a.run.app/mcp,protocolBinding=JSONRPC
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real


## Phase 3 — Bindings & endpoints (authorize connections)
📖 [debugger reference](https://github.com/jswortz/hub-spoke-agents-gcp-26)

A **Binding** authorizes a *source* (an agent) to reach a *target* (an MCP server, another agent, or an API endpoint), identified by **URN**. This is how you restrict which tools an agent may use. Listing is read-only; creating a binding is guarded.

In [4]:
# Read-only: what's registered in this region (attempted live).
run_or_show(["gcloud", "alpha", "agent-registry", "mcp-servers", "list",
             "--project", HUB_PROJECT, "--location", REGION], live=True, timeout=60)

# Guarded: authorize a source agent -> target MCP server (URN grammar).
run_or_show([
    "gcloud", "alpha", "agent-registry", "bindings", "create", "coordinator-to-search",
    "--project", HUB_PROJECT, "--location", REGION,
    "--source-identifier=urn:agent:projects-<HUB_NUM>:projects:<HUB_NUM>:locations:" + REGION + ":aiplatform:reasoningEngines:<ENGINE_ID>",
    "--target-identifier=urn:mcp:projects-<HUB_NUM>:projects:<HUB_NUM>:locations:" + REGION + ":mcpServers:<MCP_ID>",
], RUN_REGISTRY)

$ gcloud alpha agent-registry mcp-servers list --project wortz-project-352116 --location us-central1


ist all bookings for a user
  name: list_bookings
updateTime: '2026-05-16T00:33:06.398590Z'
---
attributes:
  agentregistry.googleapis.com/system/RuntimeReference:
    uri: //agentregistry.googleapis.com/projects/679926387543/locations/us-central1/services/expense-mcp
createTime: '2026-05-16T00:33:09.298451Z'
displayName: expense-mcp
interfaces:
- protocolBinding: JSONRPC
  url: https://expense-mcp-in2bk2mdwa-uc.a.run.app/mcp
mcpServerId: urn:mcp:projects-679926387543:projects:679926387543:locations:us-central1:agentregistry:services:expense-mcp
name: projects/wortz-project-352116/locations/us-central1/mcpServers/agentregistry-00000000-0000-0000-c898-ba4f1d648546
tools:
- annotations:
    readOnlyHint: true
  description: Check if an expense amount is within corporate policy for a category
  name: check_expense_policy
- description: Submit an expense report for a user
  name: submit_expense
- annotations:
    readOnlyHint: true
  description: Get all expenses for a user
  name: get_use

## Phase 4 — Hub-and-spoke discovery across projects 🌟
📖 [`jswortz/hub-spoke-agents-gcp-26`](https://github.com/jswortz/hub-spoke-agents-gcp-26)

The goal: deploy an agent in the **spoke** (`agent-spoke`) and have the **hub** discover and search it across the project boundary. **App Hub** establishes the boundary (host + service projects); once attached, App Hub *automatically* discovers the spoke's Reasoning Engines, and Agent Registry **`search`** (not `list`) queries across it. Two boundary options: **A** folder-level (auto, GCP creates a `google-mpf-*` management project) or **B** project-level (manually attach the spoke).

In [5]:
# (1) Enable the APIs in BOTH hub and spoke (guarded):
run_or_show(["gcloud", "services", "enable", "apphub.googleapis.com",
             "agentregistry.googleapis.com", "aiplatform.googleapis.com", "--project", HUB_PROJECT], RUN_REGISTRY)

# (2) Option B — attach the spoke project to the hub's App Hub boundary (guarded):
run_or_show(["gcloud", "apphub", "service-projects", "add", SPOKE_PROJECT, "--project", HUB_PROJECT], RUN_REGISTRY)

#     (Option A alternative: enable Application Management on the parent FOLDER — no attach needed;
#      GCP provisions a system-managed 'google-mpf-*' host project and auto-attaches every project in it.)

$ gcloud services enable apphub.googleapis.com agentregistry.googleapis.com aiplatform.googleapis.com --project wortz-project-352116
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real
$ gcloud apphub service-projects add agent-spoke --project wortz-project-352116
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real


In [6]:
# (3) Deploy the agent in the SPOKE project (guarded, reference command):
run_or_show(f"gcloud config set project {SPOKE_PROJECT} && python -m src.deploy.deploy_agents coordinator", RUN_REGISTRY)

# (4) Verify App Hub discovered the spoke's Reasoning Engine (read-only, attempted live):
run_or_show(["gcloud", "apphub", "discovered-workloads", "list",
             "--project", HUB_PROJECT, "--location", REGION,
             "--filter", "workloadReference.uri:aiplatform.googleapis.com"], live=True, timeout=90)

$ gcloud config set project agent-spoke && python -m src.deploy.deploy_agents coordinator
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real
$ gcloud apphub discovered-workloads list --project wortz-project-352116 --location us-central1 --filter workloadReference.uri:aiplatform.googleapis.com


rtz-project-352116', 'location': 'us-central1'}
apphub-00000000-0000-0000-634b-37542a830019  {'uri': '//aiplatform.googleapis.com/projects/679926387543/locations/us-central1/tuningJobs/7610271638617063424'}           {'gcpProject': 'projects/wortz-project-352116', 'location': 'us-central1'}
apphub-00000000-0000-0000-f57a-94adbd4602ed  {'uri': '//aiplatform.googleapis.com/projects/679926387543/locations/us-central1/tuningJobs/8081460749630701568'}           {'gcpProject': 'projects/wortz-project-352116', 'location': 'us-central1'}
apphub-00000000-0000-0000-54f8-f81ac1e47d02  {'uri': '//aiplatform.googleapis.com/projects/679926387543/locations/us-central1/tuningJobs/8191770353198956544'}           {'gcpProject': 'projects/wortz-project-352116', 'location': 'us-central1'}
apphub-00000000-0000-0000-3bd7-1e8a78989fa9  {'uri': '//aiplatform.googleapis.com/projects/679926387543/locations/us-central1/tuningJobs/8587552757756461056'}           {'gcpProject': 'projects/wortz-project-352116', 'lo

CompletedProcess(args=['gcloud', 'apphub', 'discovered-workloads', 'list', '--project', 'wortz-project-352116', '--location', 'us-central1', '--filter', 'workloadReference.uri:aiplatform.googleapis.com'], returncode=0, stdout="ID                                           WORKLOAD_REFERENCE                                                                                                          WORKLOAD_PROPERTIES\napphub-00000000-0000-0000-c4c6-b0ea0dbd1bb3  {'uri': '//aiplatform.googleapis.com/projects/371207989051/locations/us-central1/reasoningEngines/154554501225775104'}      {'extendedMetadata': {'apphub.googleapis.com/AgentProperties': {'metadataStruct': {'description': 'NovaStorm DSSIB agent with synthetic retail dataset support', 'displayName': 'novastorm-20260617220320', 'framework': 'google-adk', 'protocols': [{'interfaces': [{'protocolBinding': 'HTTP_JSON', 'url': 'https://us-central1-aiplatform.googleapis.com/v1/projects/371207989051/locations/us-central1/reasoningEngines/15

In [7]:
# (5) The payoff — SEARCH the registry across the boundary (read-only, attempted live).
#     'search' returns local + discovered spoke agents; 'list' would show only local ones.
#     Single region only — wildcards like --location=- are NOT supported.
run_or_show(["gcloud", "alpha", "agent-registry", "agents", "search",
             "--project", HUB_PROJECT, "--location", REGION], live=True, timeout=90)

$ gcloud alpha agent-registry agents search --project wortz-project-352116 --location us-central1


is.com/system/RuntimeReference:
      uri: //aiplatform.googleapis.com/projects/679926387543/locations/us-central1/reasoningEngines/5895016748914049024
  createTime: '2026-06-30T16:46:06.819926Z'
  displayName: coordinator_agent
  name: projects/wortz-project-352116/locations/us-central1/agents/agentregistry-00000000-0000-0000-24f9-402970f723f6
  protocols:
  - interfaces:
    - protocolBinding: HTTP_JSON
      url: https://us-central1-aiplatform.googleapis.com/v1/projects/679926387543/locations/us-central1/reasoningEngines/5895016748914049024:query
    - protocolBinding: HTTP_JSON
      url: https://us-central1-aiplatform.googleapis.com/v1/projects/679926387543/locations/us-central1/reasoningEngines/5895016748914049024:streamQuery
    type: CUSTOM
  uid: agentregistry-00000000-0000-0000-24f9-402970f723f6
  updateTime: '2026-06-30T16:46:06.819926Z'
- agentId: urn:agent:googleapis.com:locations:global:workspaceagent:workspaceagent--a2a
  createTime: '2026-02-06T16:37:41.413148Z'
  descr

CompletedProcess(args=['gcloud', 'alpha', 'agent-registry', 'agents', 'search', '--project', 'wortz-project-352116', '--location', 'us-central1'], returncode=0, stdout="agents:\n- agentId: urn:agent:projects-371207989051:projects:371207989051:locations:us-central1:aiplatform:reasoningEngines:154554501225775104\n  attributes:\n    agentregistry.googleapis.com/system/Framework:\n      framework: google-adk\n    agentregistry.googleapis.com/system/RuntimeIdentity:\n      principal: sa://service-371207989051@gcp-sa-aiplatform-re.iam.gserviceaccount.com\n    agentregistry.googleapis.com/system/RuntimeReference:\n      uri: //aiplatform.googleapis.com/projects/371207989051/locations/us-central1/reasoningEngines/154554501225775104\n  createTime: '2026-06-17T22:07:23.594660Z'\n  description: NovaStorm DSSIB agent with synthetic retail dataset support\n  displayName: novastorm-20260617220320\n  name: projects/wortz-project-352116/locations/us-central1/agents/agentregistry-00000000-0000-0000-c4c

In [8]:
# (6) Group the discovered spoke workload into an App Hub Application (guarded):
run_or_show(["gcloud", "apphub", "applications", "create", "payroll-app",
             "--location", REGION, "--scope-type=REGIONAL", "--project", HUB_PROJECT], RUN_REGISTRY)
run_or_show(["gcloud", "apphub", "applications", "workloads", "create", "coordinator-workload",
             "--application=payroll-app", "--location", REGION,
             "--discovered-workload=<DISCOVERED_WORKLOAD_ID>", "--project", HUB_PROJECT], RUN_REGISTRY)

$ gcloud apphub applications create payroll-app --location us-central1 --scope-type=REGIONAL --project wortz-project-352116
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real
$ gcloud apphub applications workloads create coordinator-workload --application=payroll-app --location us-central1 '--discovered-workload=<DISCOVERED_WORKLOAD_ID>' --project wortz-project-352116
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real


## Phase 5 — Centralized governance & cross-project observability
📖 [`hub-spoke-agents-gcp-26` §Governance/Observability](https://github.com/jswortz/hub-spoke-agents-gcp-26)

Once spoke agents are discovered in the hub, you govern them centrally: **IAM/IdP groups** (invocation control), **Bindings** (which tools/agents each may reach), **Auth Providers** (delegated credentials in the hub, not the spoke), centralized **Model Armor** (hub templates enforced by the spoke's gateway), and **cross-project metrics scoping** so the hub's Cloud Monitoring can alert on spoke agent latency/errors.

In [9]:
# Add the spoke to the hub's metrics scope so hub dashboards/alerts see spoke Reasoning Engine metrics:
run_or_show(["gcloud", "beta", "monitoring", "metrics-scopes", "create",
             f"projects/{SPOKE_PROJECT}", "--project", HUB_PROJECT], RUN_REGISTRY)

# Auth Providers for delegated credentials live in the hub (read-only, attempted live):
run_or_show(["gcloud", "alpha", "agent-identity", "auth-providers", "list",
             "--project", HUB_PROJECT, "--location", REGION], RUN_REGISTRY)

print("\nNote: custom agent attributes (e.g. team/org labels) are NOT surfaced for implicitly-discovered")
print("agents — only system attributes (Framework, RuntimeIdentity, RuntimeReference) are populated.")

$ gcloud beta monitoring metrics-scopes create projects/agent-spoke --project wortz-project-352116
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real
$ gcloud alpha agent-identity auth-providers list --project wortz-project-352116 --location us-central1
  🔒 skipped — set GEAP_RUN_REGISTRY=1 to run this for real

Note: custom agent attributes (e.g. team/org labels) are NOT surfaced for implicitly-discovered
agents — only system attributes (Framework, RuntimeIdentity, RuntimeReference) are populated.


## Recap

- **Register** Services from toolspecs; discover with `get_mcp_tools` / `AgentRegistry.get_mcp_toolset` (✅).
- **Authorize** connections with Bindings (source→target URN).
- **Hub-and-spoke** (🌟): App Hub boundary (`apphub service-projects add agent-spoke`) → automatic discovery (`discovered-workloads list`) → cross-boundary **`agent-registry agents search`** → group into an App Hub Application. Reads are live; boundary changes are guarded behind `GEAP_RUN_REGISTRY=1`.
- **Govern centrally** from the hub: bindings, auth providers, Model Armor, cross-project metrics scoping.

Reference: [`jswortz/hub-spoke-agents-gcp-26`](https://github.com/jswortz/hub-spoke-agents-gcp-26) (`verify_setup.sh`, `agents.md`). Deploy + monitor an MCP server itself in **`mcp_sdk_demo.ipynb`**.